# CHAPTER 7 反向传播算法：如何高效地求解梯度？

在第五章和第六章的内容当中，我们以梯度下降算法为核心进行讲解。梯度下降算法告诉了我们模型的参数应该怎么去优化，通过对损失函数求梯度，对所有参数进行更新。

但是我们所有的举得例子都是单隐藏层的例子，我们不需要对参数求二阶导甚至更高阶的导数就能完成对参数的优化。但是在多层次的神经网络当中，参数成千上万是相当正常的事情。而且隐藏层数增多以后，对于参数的求导涉及很长的链式法则。

为了提高计算梯度的效率，我们提出了反向传播算法，正是有了这种算法，我们才能够对庞大的神经网络进行训练。

所以，梯度下降和反向传播是相辅相成的，它们共同完成了对参数的优化。

## 7.1 符号约定与前置知识回顾

在正式进入反向传播之前，我们先回顾第二章建立的前向传播符号体系。设网络共有 $L$ 层（输入不算一层，$\mathbf{a}^{(0)} = \mathbf{x}$），第 $l$ 层（$l = 1, 2, \ldots, L$）的前向传播为：

$$
\begin{aligned}
\mathbf{z}^{(l)} &= \mathbf{W}^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)} \quad &\text{(线性变换)} \\[4pt]
\mathbf{a}^{(l)} &= h\bigl(\mathbf{z}^{(l)}\bigr) \quad &\text{(激活函数)}
\end{aligned}
$$

<img src="../attachment/图7-1.jpg" width="700" style="display: block; margin: 0 auto;" >

其中：
- $\mathbf{W}^{(l)} \in \mathbb{R}^{n_l \times n_{l-1}}$：第 $l$ 层的权重矩阵，$w_{ji}^{(l)}$ 表示从第 $l-1$ 层的第 $i$ 个神经元连接到第 $l$ 层第 $j$ 个神经元的权重
- $\mathbf{b}^{(l)} \in \mathbb{R}^{n_l}$：偏置向量
- $\mathbf{z}^{(l)} \in \mathbb{R}^{n_l}$：激活前的"净输入"（weighted input）
- $\mathbf{a}^{(l)} \in \mathbb{R}^{n_l}$：激活后的输出
- $h(\cdot)$：激活函数（如 ReLU、Sigmoid 等）

对于单个训练样本 $(\mathbf{x}, \mathbf{y})$，记损失函数为 $\mathcal{L}\bigl(\mathbf{a}^{(L)}, \mathbf{y}\bigr)$，简记为 $\mathcal{L}$。我们的目标是：**高效地求出 $\mathcal{L}$ 对每一层 $\mathbf{W}^{(l)}$ 和 $\mathbf{b}^{(l)}$ 的梯度**。

## 7.2 中间变量 $\delta$：反向传播的"关键枢纽"

直接对 $\mathbf{W}^{(l)}$ 求导需要展开从输出层到第 $l$ 层的漫长链式法则，极为繁琐。反向传播的精妙之处在于引入了一个**中间变量**——**误差信号**（error signal）$\boldsymbol{\delta}^{(l)}$：

> $$
> \boxed{\boldsymbol{\delta}^{(l)} \equiv \frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(l)}}}
> $$

这是一个 $n_l \times 1$ 的列向量，第 $j$ 个分量 $\delta_j^{(l)} = \partial\mathcal{L} / \partial z_j^{(l)}$ 表示：第 $l$ 层第 $j$ 个神经元**净输入的一个微小变化**，会给最终损失带来多大的影响。等价地写为梯度记号：

$$
\boldsymbol{\delta}^{(l)} = \nabla_{\mathbf{z}^{(l)}} \mathcal{L}
$$

## 7.3 四个核心方程：以 $\delta$ 为纽带的完整推导

以 $\boldsymbol{\delta}^{(l)}$ 为核心，整个反向传播算法可以凝练为**四个方程**。这些方程的编号（BP1–BP4）源自经典文献，我们逐一推导。

---

### 方程 BP1：输出层误差（Base Case）

> $$
> \boxed{\boldsymbol{\delta}^{(L)} = \nabla_{\mathbf{a}^{(L)}} \mathcal{L} \;\odot\; h'\bigl(\mathbf{z}^{(L)}\bigr)}
> \tag{BP1}
> $$

**推导**：由定义，$\boldsymbol{\delta}^{(L)}$ 是 $\mathcal{L}$ 对 $\mathbf{z}^{(L)}$ 的梯度，写成分量形式的列向量：

$$
\boldsymbol{\delta}^{(L)} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(L)}}
= \begin{pmatrix}
\dfrac{\partial \mathcal{L}}{\partial z_1^{(L)}} \\[6pt]

\dfrac{\partial \mathcal{L}}{\partial z_2^{(L)}} \\[2pt]

\vdots \\[2pt]
\dfrac{\partial \mathcal{L}}{\partial z_{n_L}^{(L)}}

\end{pmatrix}
$$

对第 $j$ 个分量应用链式法则（以 $a_j^{(L)}$ 为中间变量）。由于 $\mathbf{a}^{(L)} = h(\mathbf{z}^{(L)})$ 是逐元素操作，$a_j^{(L)} = h(z_j^{(L)})$ 仅依赖于 $z_j^{(L)}$：

$$
\frac{\partial \mathcal{L}}{\partial z_j^{(L)}}
= \frac{\partial \mathcal{L}}{\partial a_j^{(L)}} \cdot \frac{\partial a_j^{(L)}}{\partial z_j^{(L)}}
= \frac{\partial \mathcal{L}}{\partial a_j^{(L)}} \cdot h'\bigl(z_j^{(L)}\bigr)
$$

将所有分量收集回列向量，两个列向量的逐元素相乘正是 Hadamard 积 $\odot$：

$$
\boldsymbol{\delta}^{(L)}
= \begin{pmatrix}
\dfrac{\partial \mathcal{L}}{\partial a_1^{(L)}} \\[6pt]
\dfrac{\partial \mathcal{L}}{\partial a_2^{(L)}} \\[2pt]
\vdots \\[2pt]
\dfrac{\partial \mathcal{L}}{\partial a_{n_L}^{(L)}}
\end{pmatrix}
\;\odot\;
\begin{pmatrix}
h'(z_1^{(L)}) \\[2pt]
h'(z_2^{(L)}) \\[2pt]
\vdots \\[2pt]
h'(z_{n_L}^{(L)})
\end{pmatrix}
= \nabla_{\mathbf{a}^{(L)}} \mathcal{L} \;\odot\; h'\bigl(\mathbf{z}^{(L)}\bigr)
$$


**BP1 是反向传播的起点**——它是唯一一个"不依赖其他层 $\delta$"的方程。在递归的语境中，BP1 就是**基案（Base Case）**：当 $l = L$ 时，我们直接从损失函数算出 $\boldsymbol{\delta}^{(L)}$，无需递归。

### 方程 BP2：误差的反向传播（Recursive Step）

> $$
> \boxed{\boldsymbol{\delta}^{(l-1)} = \Bigl(\mathbf{W}^{(l)}\Bigr)^{\!T} \boldsymbol{\delta}^{(l)} \;\odot\; h'\bigl(\mathbf{z}^{(l-1)}\bigr)}
> \tag{BP2}
> $$

**这是四个方程中最核心的一个。** BP2 定义了如何从第 $l$ 层的误差 $\boldsymbol{\delta}^{(l)}$ 算出前一层的误差 $\boldsymbol{\delta}^{(l-1)}$——**这就是反向传播中"传播"二字的数学本质，也是递归关系的灵魂所在。** 注意到 BP2 对 $l = L, L-1, \ldots, 2$ 都成立：从 $\boldsymbol{\delta}^{(L)}$（由 BP1 给出）出发，依次取 $l = L$ 得 $\boldsymbol{\delta}^{(L-1)}$，取 $l = L-1$ 得 $\boldsymbol{\delta}^{(L-2)}$，……直至 $\boldsymbol{\delta}^{(1)}$。

<img src="../attachment/图7-2.jpg" width="700" style="display: block; margin: 0 auto;" >

**推导**：从分量出发。由定义 $\delta_j^{(l-1)} = \dfrac{\partial \mathcal{L}}{\partial z_j^{(l-1)}}$。关键在于：第 $l-1$ 层的 $z_j^{(l-1)}$ 会影响第 $l$ 层的**每一个** $z_k^{(l)}$——先经过 $h$ 变成 $a_j^{(l-1)}$，再通过权重 $w_{kj}^{(l)}$ 流向所有 $z_k^{(l)}$。因此以全部 $\{z_k^{(l)}\}_{k=1}^{n_l}$ 为中间变量，应用多元链式法则对 $k$ 求和：

$$
\delta_j^{(l-1)}
= \sum_{k=1}^{n_l} \frac{\partial \mathcal{L}}{\partial z_k^{(l)}} \cdot \frac{\partial z_k^{(l)}}{\partial z_j^{(l-1)}}
= \sum_{k=1}^{n_l} \delta_k^{(l)} \cdot \frac{\partial z_k^{(l)}}{\partial z_j^{(l-1)}}
\tag{1}
$$

现在计算关键偏导数 $\dfrac{\partial z_k^{(l)}}{\partial z_j^{(l-1)}}$。在前向传播中：

$$
z_k^{(l)} = \sum_{i=1}^{n_{l-1}} w_{ki}^{(l)} \, h\bigl(z_i^{(l-1)}\bigr) + b_k^{(l)}
$$

对 $z_j^{(l-1)}$ 求偏导——注意求和中仅 $i = j$ 那一项含有 $z_j^{(l-1)}$（因为 $h$ 是逐元素函数），其余 $i \neq j$ 项全部为零：

$$
\frac{\partial z_k^{(l)}}{\partial z_j^{(l-1)}}
= w_{kj}^{(l)} \cdot h'\bigl(z_j^{(l-1)}\bigr)
\tag{2}
$$

将 $(2)$ 代入 $(1)$：

$$
\delta_j^{(l-1)}
= \sum_{k=1}^{n_l} \delta_k^{(l)} \cdot w_{kj}^{(l)} \cdot h'\bigl(z_j^{(l-1)}\bigr)
$$

$h'(z_j^{(l-1)})$ 不依赖于求和指标 $k$，提出求和号：

$$
\delta_j^{(l-1)}
= \left( \sum_{k=1}^{n_l} w_{kj}^{(l)} \, \delta_k^{(l)} \right)\;\cdot\; h'\bigl(z_j^{(l-1)}\bigr)
\tag{3}
$$

---


**从分量到列向量**。将 $(3)$ 对所有 $j = 1, 2, \ldots, n_{l-1}$ 同时写出，得到完整的列向量 $\boldsymbol{\delta}^{(l-1)}$：

$$
\boldsymbol{\delta}^{(l-1)}
= \begin{pmatrix}
\delta_1^{(l-1)} \\[4pt]
\delta_2^{(l-1)} \\[2pt]
\vdots \\[2pt]
\delta_{n_{l-1}}^{(l-1)}
\end{pmatrix}
= \begin{pmatrix}
\left( \displaystyle\sum_{k=1}^{n_l} w_{k1}^{(l)} \delta_k^{(l)} \right) \cdot h'(z_1^{(l-1)}) \\[12pt]
\left( \displaystyle\sum_{k=1}^{n_l} w_{k2}^{(l)} \delta_k^{(l)} \right) \cdot h'(z_2^{(l-1)}) \\[10pt]
\vdots \\[10pt]
\left( \displaystyle\sum_{k=1}^{n_l} w_{k n_{l-1}}^{(l)} \delta_k^{(l)} \right) \cdot h'(z_{n_{l-1}}^{(l-1)})
\end{pmatrix}
$$

将每行的乘积拆为 Hadamard 积的两个列向量：

$$
= \begin{pmatrix}
\displaystyle\sum_{k=1}^{n_l} w_{k1}^{(l)} \delta_k^{(l)} \\[10pt]
\displaystyle\sum_{k=1}^{n_l} w_{k2}^{(l)} \delta_k^{(l)} \\[8pt]
\vdots \\[8pt]
\displaystyle\sum_{k=1}^{n_l} w_{k n_{l-1}}^{(l)} \delta_k^{(l)}
\end{pmatrix}
\;\odot\;
\begin{pmatrix}
h'(z_1^{(l-1)}) \\[4pt]
h'(z_2^{(l-1)}) \\[2pt]
\vdots \\[2pt]
h'(z_{n_{l-1}}^{(l-1)})
\end{pmatrix}
\tag{4}
$$

---


**从求和到转置矩阵**。关键一步：识别分量 ① 中的求和。展开第一个分量看：

$$
\sum_{k=1}^{n_l} w_{k1}^{(l)} \delta_k^{(l)}
= w_{11}^{(l)} \delta_1^{(l)} + w_{21}^{(l)} \delta_2^{(l)} + \cdots + w_{n_l 1}^{(l)} \delta_{n_l}^{(l)}
$$

这是用矩阵的第 1 列（$\mathbf{W}^{(l)}$ 中所有行在第 1 列上的元素）与向量 $\boldsymbol{\delta}^{(l)}$ 做内积。推广到所有 $j$，整个列向量 ① 就是用一个矩阵去乘 $\boldsymbol{\delta}^{(l)}$，该矩阵的**第 $j$ 行第 $k$ 列**恰好是 $w_{kj}^{(l)}$——注意指标顺序！这正说明这个矩阵是 $\mathbf{W}^{(l)}$ 的转置：

$$
\begin{pmatrix}
\sum_k w_{k1}^{(l)} \delta_k^{(l)} \\[6pt]
\sum_k w_{k2}^{(l)} \delta_k^{(l)} \\[4pt]
\vdots \\[4pt]
\sum_k w_{k n_{l-1}}^{(l)} \delta_k^{(l)}
\end{pmatrix}
=
\underbrace{\begin{pmatrix}
w_{11}^{(l)} & w_{21}^{(l)} & \cdots & w_{n_l 1}^{(l)} \\[3pt]
w_{12}^{(l)} & w_{22}^{(l)} & \cdots & w_{n_l 2}^{(l)} \\[2pt]
\vdots & \vdots & \ddots & \vdots \\[2pt]
w_{1 n_{l-1}}^{(l)} & w_{2 n_{l-1}}^{(l)} & \cdots & w_{n_l n_{l-1}}^{(l)}
\end{pmatrix}}_{(\mathbf{W}^{(l)})^T \;\in\; \mathbb{R}^{n_{l-1} \times n_l}}
\begin{pmatrix}
\delta_1^{(l)} \\[4pt]
\delta_2^{(l)} \\[2pt]
\vdots \\[2pt]
\delta_{n_l}^{(l)}
\end{pmatrix}
= \Bigl(\mathbf{W}^{(l)}\Bigr)^{\!T} \boldsymbol{\delta}^{(l)}
\tag{5}
$$

将 $(5)$ 代入 $(4)$，即得 BP2 的最终形式：

$$
\boxed{\boldsymbol{\delta}^{(l-1)} = \Bigl(\mathbf{W}^{(l)}\Bigr)^{\!T} \boldsymbol{\delta}^{(l)} \;\odot\; h'\bigl(\mathbf{z}^{(l-1)}\bigr)}
$$


---

### 深度解读 BP2：它到底在做什么？

把 BP2 拆成两步来理解：

$$
\underbrace{\Bigl(\mathbf{W}^{(l)}\Bigr)^{\!T} \boldsymbol{\delta}^{(l)}}_{\text{Step 1: 误差回传}}
\;\odot\;
\underbrace{h'\bigl(\mathbf{z}^{(l-1)}\bigr)}_{\text{Step 2: 门控修正}}
$$

- **Step 1 — 误差回传**：$\bigl(\mathbf{W}^{(l)}\bigr)^T \boldsymbol{\delta}^{(l)}$ 把第 $l$ 层的误差信号沿着权重矩阵的转置"反向输送"回第 $l-1$ 层。这正是上节推导中从求和 $\sum_k w_{kj}^{(l)} \delta_k^{(l)}$ 识别出 $(\mathbf{W}^{(l)})^T \boldsymbol{\delta}^{(l)}$ 的几何含义——前向传播用 $\mathbf{W}^{(l)}$ 将信息**发散**，反向传播用其**转置**将误差重新**汇聚**。

- **Step 2 — 门控修正**：乘上 $h'(\mathbf{z}^{(l-1)})$ 是对误差信号的一次"筛选"。以 ReLU 为例，$h'(z) = 1$ 当 $z > 0$，否则为 $0$——**在 ReLU 关断的神经元上，误差信号被完全阻断**（这正是"Dying ReLU"现象的数学根源）。以 Sigmoid 为例，$h'(z)$ 在 $|z|$ 很大时趋于 $0$，误差信号衰减，从而导致梯度消失。

### 方程 BP3 & BP4：从误差信号到参数梯度

有了 $\boldsymbol{\delta}^{(l)}$，对参数 $\mathbf{b}^{(l)}$ 和 $\mathbf{W}^{(l)}$ 的梯度几乎唾手可得。

---

#### 方程 BP3：偏置梯度

> $$
> \boxed{\nabla_{\mathbf{b}^{(l)}} \mathcal{L} = \boldsymbol{\delta}^{(l)}}
> \tag{BP3}
> $$

**推导**：由 $\mathbf{z}^{(l)} = \mathbf{W}^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}$，写下 Jacobian 矩阵 $\dfrac{\partial \mathbf{z}^{(l)}}{\partial \mathbf{b}^{(l)}}$：

$$
\frac{\partial z_j^{(l)}}{\partial b_k^{(l)}} = \delta_{jk} = \begin{cases} 1 & j = k \\ 0 & j \neq k \end{cases}
\quad\Longrightarrow\quad
\frac{\partial \mathbf{z}^{(l)}}{\partial \mathbf{b}^{(l)}} = \mathbf{I}_{n_l}
$$

因此：

$$
\nabla_{\mathbf{b}^{(l)}} \mathcal{L}
= \left( \frac{\partial \mathbf{z}^{(l)}}{\partial \mathbf{b}^{(l)}} \right)^{\!T} \frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(l)}}
= \mathbf{I}_{n_l} \cdot \boldsymbol{\delta}^{(l)}
= \boldsymbol{\delta}^{(l)}
$$

> 这是四个方程中最简洁的一个。**误差有多灵敏，偏置就该调多少。**


---

#### 方程 BP4：权重梯度

> $$
> \boxed{\nabla_{\mathbf{W}^{(l)}} \mathcal{L} = \boldsymbol{\delta}^{(l)} \bigl(\mathbf{a}^{(l-1)}\bigr)^T}
> \tag{BP4}
> $$

**推导**：由 $z_j^{(l)} = \sum_{i=1}^{n_{l-1}} w_{ji}^{(l)} a_i^{(l-1)} + b_j^{(l)}$，对单个权重 $w_{ji}^{(l)}$ 求偏导：

$$
\frac{\partial \mathcal{L}}{\partial w_{ji}^{(l)}}
= \frac{\partial \mathcal{L}}{\partial z_j^{(l)}} \cdot \frac{\partial z_j^{(l)}}{\partial w_{ji}^{(l)}}
= \delta_j^{(l)} \cdot a_i^{(l-1)}
$$

将所有 $(j, i)$ 对排成 $n_l \times n_{l-1}$ 矩阵：

$$
\nabla_{\mathbf{W}^{(l)}} \mathcal{L}
= \begin{pmatrix}
\delta_1^{(l)} a_1^{(l-1)} & \delta_1^{(l)} a_2^{(l-1)} & \cdots & \delta_1^{(l)} a_{n_{l-1}}^{(l-1)} \\[3pt]
\delta_2^{(l)} a_1^{(l-1)} & \delta_2^{(l)} a_2^{(l-1)} & \cdots & \delta_2^{(l)} a_{n_{l-1}}^{(l-1)} \\[2pt]
\vdots & \vdots & \ddots & \vdots \\[2pt]
\delta_{n_l}^{(l)} a_1^{(l-1)} & \delta_{n_l}^{(l)} a_2^{(l-1)} & \cdots & \delta_{n_l}^{(l)} a_{n_{l-1}}^{(l-1)}
\end{pmatrix}
= \underbrace{\begin{pmatrix} \delta_1^{(l)} \\ \delta_2^{(l)} \\ \vdots \\ \delta_{n_l}^{(l)} \end{pmatrix}}_{n_l \times 1}
\;\cdot\;
\underbrace{\begin{pmatrix} a_1^{(l-1)} & a_2^{(l-1)} & \cdots & a_{n_{l-1}}^{(l-1)} \end{pmatrix}}_{1 \times n_{l-1}}
$$

这正是**外积**（outer product）$\boldsymbol{\delta}^{(l)} (\mathbf{a}^{(l-1)})^T$——列向量 $\boldsymbol{\delta}^{(l)}$ 与行向量 $(\mathbf{a}^{(l-1)})^T$ 相乘，得到 $n_l \times n_{l-1}$ 矩阵，维度与 $\mathbf{W}^{(l)}$ 完全一致。

> **BP4 = "错误信号 × 输入强度"。** $\delta_j^{(l)}$ 大说明神经元 $j$ 的误差敏感度高，$a_i^{(l-1)}$ 大说明输入信号强——两者的乘积 $a_i^{(l-1)} \delta_j^{(l)}$ 决定了权重 $w_{ji}^{(l)}$ 的修正幅度。若输入为零，即便有误差也不关这条连接的事，权重无需调整。

---

### 🧩 四方程全景：一张图看清它们如何协作

将四个方程放在一起，$\boldsymbol{\delta}$ 的"桥梁"作用一目了然：

| 方程 | 表达式 | 角色 |
|:---:|:---|:---|
| **BP1** | $\boldsymbol{\delta}^{(L)} = \nabla_{\mathbf{a}^{(L)}} \mathcal{L} \odot h'(\mathbf{z}^{(L)})$ | **基案**：从损失直接算输出层 $\delta$ |
| **BP2** | $\boldsymbol{\delta}^{(l-1)} = (\mathbf{W}^{(l)})^T \boldsymbol{\delta}^{(l)} \odot h'(\mathbf{z}^{(l-1)})$ | **递归**：从 $\delta^{(l)}$ 推算 $\delta^{(l-1)}$ |
| **BP3** | $\nabla_{\mathbf{b}^{(l)}} \mathcal{L} = \boldsymbol{\delta}^{(l)}$ | **产出**：由 $\delta$ 得到偏置梯度 |
| **BP4** | $\nabla_{\mathbf{W}^{(l)}} \mathcal{L} = \boldsymbol{\delta}^{(l)} (\mathbf{a}^{(l-1)})^T$ | **产出**：由 $\delta$ 得到权重梯度 |

四个方程的协作方式如下——BP1 在输出层点燃火种，此后每一步取 BP2 中 $l = L, L-1, \ldots, 2$，逐层将误差反向传播，BP3 和 BP4 在每一站收获梯度：

> **BP1 点燃火种，BP2 层层传递，BP3 和 BP4 在每一站收获果实。**

$$
\underbrace{\boldsymbol{\delta}^{(L)}}_{\text{BP1}}
\;\xrightarrow{\text{BP2}\; l=L}\;
\underbrace{\boldsymbol{\delta}^{(L-1)}}_{\text{BP3,4}\;\Rightarrow\;\nabla\mathbf{W}^{(L-1)},\nabla\mathbf{b}^{(L-1)}}
\;\xrightarrow{\text{BP2}\; l=L-1}\;
\underbrace{\boldsymbol{\delta}^{(L-2)}}_{\text{BP3,4}\;\Rightarrow\;\nabla\mathbf{W}^{(L-2)},\nabla\mathbf{b}^{(L-2)}}
\;\xrightarrow{\text{BP2}}\;
\cdots
\;\xrightarrow{\text{BP2}\; l=2}\;
\underbrace{\boldsymbol{\delta}^{(1)}}_{\text{BP3,4}\;\Rightarrow\;\nabla\mathbf{W}^{(1)},\nabla\mathbf{b}^{(1)}}
$$

## 7.4 递归：反向传播的算法之魂

现在让我们从算法的视角重新审视这四个方程。当你把 BP1、BP2、BP3、BP4 放在一起看，一个**递归结构**（recursive structure）跃然纸上。

### 7.4.1 为何说反向传播是递归算法？

递归算法的两个必备要素，在反向传播中恰好一一对应：

| 递归要素 | 反向传播中的对应 |
|:---|:---|
| **基案（Base Case）**：可直接求解、无需递归的情况 | **BP1**：$\boldsymbol{\delta}^{(L)}$ 直接从损失函数算出 |
| **递归关系（Recurrence）**：如何从子问题的解推导出当前解 | **BP2**：$\boldsymbol{\delta}^{(l-1)} = f\bigl(\boldsymbol{\delta}^{(l)}\bigr)$ |
| **回溯收集（Gather）**：在递归返回时整合结果 | **BP3 + BP4**：每层利用 $\boldsymbol{\delta}^{(l)}$ 计算本层梯度 |

用递归的伪代码来表达，这个结构更加清晰：

```
function BACKPROP(l):
    if l == L:                              // 基案（Base Case）
        δ^L = ∇_a L ⊙ h'(z^L)               // ──── BP1
        ∇W^L = δ^L · (a^{L-1})^T             // ──── BP4
        ∇b^L = δ^L                           // ──── BP3
        return [(∇W^L, ∇b^L)]
    
    else:                                    // 递归步：先向深层递归
        (δ_next, lower_grads) = BACKPROP(l + 1)
        
        // BP2: δ^{l-1} = (W^l)^T · δ^l ⊙ h'(z^{l-1})  取 l ← l+1
        δ = (W_{l+1})^T · δ_next ⊙ h'(z_l)
        //      ↑ 对应 BP2 中取 l ← l+1：δ^l = (W^{l+1})^T δ^{l+1} ⊙ h'(z^l)
        
        ∇W^l = δ · (a^{l-1})^T               // ──── BP4
        ∇b^l = δ                             // ──── BP3
        
        return (δ, [(∇W^l, ∇b^l)] + lower_grads)
```

### 7.4.2 递归的展开：一步步溯源

让我们具体追踪一个 $L=3$ 的网络，看看如何从 BP1 出发、反复套用 BP2 一步步溯源：

```
BP1 → δ^(3)                                                   // 基案：直接算出
BP2 (l=3) → δ^(2) = (W^(3))^T δ^(3) ⊙ h'(z^(2))              // δ^(3) → δ^(2)
BP2 (l=2) → δ^(1) = (W^(2))^T δ^(2) ⊙ h'(z^(1))              // δ^(2) → δ^(1)
```

每一步都是同一个 BP2 方程在起作用——取 $l = L, L-1, \ldots, 2$，将误差信号**从后往前**、一层一层地传递。每次递归返回，**只携带一个关键信息：$\boldsymbol{\delta}$**。这正是为什么 $\delta$ 被称为"中间变量"——它是在递归调用之间传递的唯一信使。

对比经典的递归算法来加深理解：

| 经典递归 | 反向传播 |
|:---|:---|
| `factorial(n) = n × factorial(n-1)` | $\boldsymbol{\delta}^{(l-1)} = (\mathbf{W}^{(l)})^T \boldsymbol{\delta}^{(l)} \odot h'(\mathbf{z}^{(l-1)})$ |
| 基案：`factorial(1) = 1` | 基案：$\boldsymbol{\delta}^{(L)} = \nabla_{\mathbf{a}^{(L)}} \mathcal{L} \odot h'(\mathbf{z}^{(L)})$ |
| 递归方向：$n \to n-1 \to \cdots \to 1$ | 递归方向：$L \to L-1 \to \cdots \to 1$ |
| 每层只传递一个整数 | 每层只传递一个向量 $\boldsymbol{\delta}$ |

> factorial 中 `n` 依赖 `n-1`，反向传播中 $\boldsymbol{\delta}^{(l-1)}$ 依赖 $\boldsymbol{\delta}^{(l)}$。两者都是"从大到小、逐步归约"的经典递归模式。

### 7.4.3 为什么这个递归是对的？——链式法则的结构性解读

反向传播能够递归，根源在于**神经网络的计算图是一个有向无环图（DAG），而损失函数对每层参数的依赖，天然形成了一条"链"**：

$$
\mathcal{L} \leftarrow \mathbf{z}^{(L)} \leftarrow \mathbf{z}^{(L-1)} \leftarrow \cdots \leftarrow \mathbf{z}^{(1)}
$$

链式法则告诉我们，对于这种链式依赖：

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(1)}}
= \frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(L)}} \cdot
\frac{\partial \mathbf{z}^{(L)}}{\partial \mathbf{z}^{(L-1)}} \cdot \;\cdots\; \cdot
\frac{\partial \mathbf{z}^{(2)}}{\partial \mathbf{z}^{(1)}}
$$

如果对每一层都从头展开这条链，中间会有大量重复计算。而 BP2 用一个**递推形式**把链式法则写成了局部迭代：

$$
\underbrace{\frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(l-1)}}}_{\boldsymbol{\delta}^{(l-1)}}
= \underbrace{\frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(l)}}}_{\boldsymbol{\delta}^{(l)}} \cdot
\;\; \frac{\partial \mathbf{z}^{(l)}}{\partial \mathbf{z}^{(l-1)}}
= (\mathbf{W}^{(l)})^T \boldsymbol{\delta}^{(l)} \odot h'(\mathbf{z}^{(l-1)})
$$

这正是**动态规划**的核心思想：存储中间结果 $\boldsymbol{\delta}^{(l)}$，避免重复展开长链条。从这个角度看：

> **反向传播 = 链式法则 + 动态规划（记忆化递归）**

这一洞见也是现代深度学习框架（PyTorch、TensorFlow、JAX）中**自动微分**（Automatic Differentiation）的理论基础。

## 7.6 小结与三层递进理解

本章我们从一篇经典推导出发，把反向传播算法的四个核心方程逐一展开，并在推导的基础上建立起了**方程之间的联系**——这种联系的关键纽带就是中间变量 $\boldsymbol{\delta}$，而这种联系的**算法本质就是递归**。

### 理解反向传播的三个层次

| 层次 | 理解方式 | 核心问题 |
|:---:|:---|:---|
| **第一层：公式层** | BP1–BP4 四个方程 | "梯度怎么算？" |
| **第二层：传播层** | $\boldsymbol{\delta}$ 如何从输出层一层一层传回输入层 | "误差怎么流？" |
| **第三层：结构层** | 链式法则 + 递归 + 动态规划 | "为什么这样设计是对的？" |

大多数教程止步于第一层——告诉你四个公式，让你套用。但真正让你**内化**反向传播的，是第二层和第三层：

- **第二层**让你看见 $\boldsymbol{\delta}$ 的流动：$\boldsymbol{\delta}^{(L)} \to \boldsymbol{\delta}^{(L-1)} \to \cdots \to \boldsymbol{\delta}^{(1)}$，每一步都通过 $(\mathbf{W}^{(l)})^T \boldsymbol{\delta}^{(l)}$ 把误差"送回上一层"，再通过 $\odot h'$ 做门控修正。
- **第三层**让你意识到：反向传播本质上就是一个**以链式法则为递推关系、以 $\boldsymbol{\delta}$ 为中间状态的记忆化递归**。BP2 中 $\boldsymbol{\delta}^{(l-1)} = f(\boldsymbol{\delta}^{(l)})$ 和 factorial 中 `F(n) = n × F(n-1)` 是同一种递归结构，只不过一个传向量、一个传整数。

### 递归视角带来的三个深层洞察

**1. 为什么梯度消失/爆炸不可避免？**

从递归的角度看，梯度消失和爆炸就是递归关系的不稳定性。BP2 中每一步递推 $\boldsymbol{\delta}^{(l-1)} = (\mathbf{W}^{(l)})^T \boldsymbol{\delta}^{(l)} \odot h'(\mathbf{z}^{(l-1)})$ 都相当于乘以一个"衰减因子"。若每一步的因子都 $< 1$（如 Sigmoid 导数最大值为 $0.25$），经过 $L$ 层递归后 $\boldsymbol{\delta}^{(1)} \propto 0.25^L \to 0$——梯度消失。反之权重过大则指数爆炸。

**2. 为什么跳跃连接（ResNet）有效？**

ResNet 引入 $\mathbf{a}^{(l)} = h(\mathbf{z}^{(l)}) + \mathbf{a}^{(l-1)}$ 的跳跃连接，等价于**在递归关系中增加了一条"高速公路"**，使得 $\boldsymbol{\delta}$ 可以绕过中间的衰减直达浅层——这与尾递归优化中使用累加器避免栈溢出的思路异曲同工。

**3. 为什么 LSTM 能处理长序列？**

LSTM 的门控机制（遗忘门、输入门、输出门）本质上是在递归关系 $\mathbf{c}_t = f_t \odot \mathbf{c}_{t-1} + i_t \odot \tilde{\mathbf{c}}_t$ 中，让"衰减因子"变得**可学习且动态可变**——在需要保持长期记忆时让 $f_t \approx 1$，误差信号就能几乎无损地传递数十甚至数百步。

### 代码中的四个方程

回顾 7.5 节的代码，你会发现四个方程在代码中只有寥寥几行：

```python
# BP1 (基案)
delta = loss_grad_fn(a_L, y) * h_prime_fn(z_L)

# BP2 (递推) — 从 δ^(l) 算 δ^(l-1)（代码中以递推形式实现：δ^l ← δ^{l+1}）
delta = weights[l].T @ delta * h_prime_fn(zs[l-1])
#       ↑ (W^{l+1})^T           ↑ ⊙ h'(z^l)

# BP4 & BP3
grad_W = delta @ a_prev.T
grad_b = delta
```

数学上的精炼直接转化为代码上的简洁——这正是反向传播算法的优雅之处。

### 与梯度下降的关系

在本章的最后，让我们回到第五、六章的梯度下降算法。反向传播解决的是"**如何高效地算出梯度**"的问题，而梯度下降（以及它的各种变体——SGD、Momentum、Adam 等）解决的是"**拿到梯度后如何更新参数**"的问题。两者分工明确、配合紧密：

$$
\boxed{\text{反向传播}} \;\xrightarrow{\text{提供 }\nabla\mathbf{W},\nabla\mathbf{b}}\; \boxed{\text{梯度下降}} \;\xrightarrow{\text{更新}}\; \boxed{\text{参数 }\mathbf{W},\mathbf{b}}
$$

掌握了反向传播的四个方程和它们的递归本质，你就掌握了深度学习训练的"动力系统"中最关键的一环。